# Train the process surrogate

Train a differentiable PyTorch model to predict quality and energy load from the five disturbances and five manipulated variables. Run this notebook from `notebooks/` after `01_generate_data.ipynb`. The chronological 70/15/15 split is made **before** fitting scalers; the test partition is evaluated only after training and early stopping.

In [ ]:
from copy import deepcopy
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.use_deterministic_algorithms(True)
DEVICE = torch.device("cpu")
DATA_PATH = Path("../data/synthetic_process_data.csv")
MODEL_DIR = Path("../models")
RESULTS_DIR = Path("../results")
INPUT_COLUMNS = [
    "d1_feed_grade", "d2_hardness", "d3_moisture", "d4_impurity", "d5_ambient",
    "u1_feed_rate", "u2_water_rate", "u3_reagent_rate", "u4_speed", "u5_temperature_setpoint",
]
OUTPUT_COLUMNS = ["y1_quality", "y2_energy_load"]
HIDDEN_SIZES = (32, 32, 16)
BATCH_SIZE = 256
MAX_EPOCHS = 150
PATIENCE = 18
MIN_DELTA = 1e-5


In [ ]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Generate the data with 01_generate_data.ipynb first: {DATA_PATH.resolve()}")
data = pd.read_csv(DATA_PATH)
required = INPUT_COLUMNS + OUTPUT_COLUMNS
missing = set(required) - set(data.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")
if len(data) != 8000 or not np.isfinite(data[required].to_numpy(dtype=float)).all():
    raise ValueError("Expected 8,000 finite, sequential observations")

train_end = int(0.70 * len(data))
val_end = int(0.85 * len(data))
partitions = {
    "train": data.iloc[:train_end],
    "validation": data.iloc[train_end:val_end],
    "test": data.iloc[val_end:],
}
input_scaler = StandardScaler().fit(partitions["train"][INPUT_COLUMNS])
output_scaler = StandardScaler().fit(partitions["train"][OUTPUT_COLUMNS])
arrays = {
    name: (
        input_scaler.transform(frame[INPUT_COLUMNS]).astype(np.float32),
        output_scaler.transform(frame[OUTPUT_COLUMNS]).astype(np.float32),
    )
    for name, frame in partitions.items()
}
print({name: len(frame) for name, frame in partitions.items()})


In [ ]:
class ProcessModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(10, 32), nn.SiLU(),
            nn.Linear(32, 32), nn.SiLU(),
            nn.Linear(32, 16), nn.SiLU(),
            nn.Linear(16, 2),
        )

    def forward(self, inputs):
        return self.network(inputs)


model = ProcessModel().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=6)
loss_fn = nn.MSELoss()
train_x, train_y = (torch.from_numpy(a).to(DEVICE) for a in arrays["train"])
val_x, val_y = (torch.from_numpy(a).to(DEVICE) for a in arrays["validation"])
train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=BATCH_SIZE,
                          shuffle=True, generator=torch.Generator().manual_seed(SEED))

history = {"train_loss": [], "validation_loss": [], "learning_rate": []}
best_loss = float("inf")
best_state = None
stale_epochs = 0
for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for features, targets in train_loader:
        optimizer.zero_grad(set_to_none=True)
        loss = loss_fn(model(features), targets)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(features)
    train_loss = total_loss / len(train_x)
    model.eval()
    with torch.no_grad():
        validation_loss = loss_fn(model(val_x), val_y).item()
    history["train_loss"].append(train_loss)
    history["validation_loss"].append(validation_loss)
    history["learning_rate"].append(optimizer.param_groups[0]["lr"])
    scheduler.step(validation_loss)
    if validation_loss < best_loss - MIN_DELTA:
        best_loss = validation_loss
        best_epoch = epoch
        best_state = deepcopy(model.state_dict())
        stale_epochs = 0
    else:
        stale_epochs += 1
        if stale_epochs >= PATIENCE:
            break
model.load_state_dict(best_state)
model.eval()
print(f"Trained {epoch} epochs; best epoch {best_epoch}, validation scaled MSE {best_loss:.6f}")


In [ ]:
predictions = {}
metrics = []
for split, (features, scaled_targets) in arrays.items():
    with torch.no_grad():
        scaled_prediction = model(torch.from_numpy(features).to(DEVICE)).cpu().numpy()
    scaled_loss = mean_squared_error(scaled_targets, scaled_prediction)
    actual = partitions[split][OUTPUT_COLUMNS].to_numpy()
    predicted = output_scaler.inverse_transform(scaled_prediction)
    predictions[split] = predicted
    for index, output in enumerate(OUTPUT_COLUMNS):
        metrics.append({
            "split": split, "output": output, "scaled_mse_loss": scaled_loss,
            "r2": r2_score(actual[:, index], predicted[:, index]),
            "mae": mean_absolute_error(actual[:, index], predicted[:, index]),
            "rmse": np.sqrt(mean_squared_error(actual[:, index], predicted[:, index])),
        })
metrics_df = pd.DataFrame(metrics)
display(metrics_df)

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
torch.save({
    "model_state_dict": model.state_dict(),
    "input_columns": INPUT_COLUMNS,
    "output_columns": OUTPUT_COLUMNS,
    "hidden_sizes": HIDDEN_SIZES,
    "best_epoch": best_epoch,
}, MODEL_DIR / "process_model.pt")
with (MODEL_DIR / "input_scaler.pkl").open("wb") as file:
    pickle.dump(input_scaler, file)
with (MODEL_DIR / "output_scaler.pkl").open("wb") as file:
    pickle.dump(output_scaler, file)
metrics_df.to_csv(RESULTS_DIR / "training_metrics.csv", index=False)
print(f"Saved model and scalers to {MODEL_DIR.resolve()}; metrics to {RESULTS_DIR.resolve()}")


## Diagnostics on the held-out test interval

Parity and residual plots use unscaled values. Residual means actual minus predicted; trends can reveal dynamic lag not captured by this static surrogate.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(history["train_loss"], label="Train")
ax.plot(history["validation_loss"], label="Validation")
ax.axvline(best_epoch - 1, color="gray", linestyle="--", label="Best epoch")
ax.set(xlabel="Epoch (zero-based)", ylabel="Scaled MSE", title="Training curves")
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "training_curves.png", dpi=150)
plt.show()


In [ ]:
test_actual = partitions["test"][OUTPUT_COLUMNS].to_numpy()
test_predicted = predictions["test"]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for index, (ax, output) in enumerate(zip(axes, OUTPUT_COLUMNS)):
    low = min(test_actual[:, index].min(), test_predicted[:, index].min())
    high = max(test_actual[:, index].max(), test_predicted[:, index].max())
    ax.scatter(test_actual[:, index], test_predicted[:, index], s=8, alpha=0.4)
    ax.plot([low, high], [low, high], "k--", linewidth=1)
    ax.set(xlabel=f"Actual {output}", ylabel=f"Predicted {output}", title=output)
fig.suptitle("Held-out test parity")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "parity_plot.png", dpi=150)
plt.show()


In [ ]:
residuals = test_actual - test_predicted
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for index, output in enumerate(OUTPUT_COLUMNS):
    axes[index, 0].scatter(test_predicted[:, index], residuals[:, index], s=8, alpha=0.4)
    axes[index, 0].axhline(0, color="black", linestyle="--", linewidth=1)
    axes[index, 0].set(xlabel=f"Predicted {output}", ylabel="Actual - predicted", title=f"{output}: residual vs prediction")
    axes[index, 1].plot(partitions["test"].index, residuals[:, index], linewidth=0.7)
    axes[index, 1].axhline(0, color="black", linestyle="--", linewidth=1)
    axes[index, 1].set(xlabel="Sequential sample", ylabel="Actual - predicted", title=f"{output}: residual over time")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "residual_plots.png", dpi=150)
plt.show()
